## The $\mathcal{VKSSM}$-EKF implementation

This cell defines the smooth surrogate transition map used by the EKF, together with its Jacobian.

The original simulator uses a hard-radius interaction rule: agents influence one another only if they lie within distance $R$. That rule is not differentiable with respect to position, since small changes in position can suddenly add or remove neighbours. For the EKF, we therefore replace the hard interaction graph by a smooth Gaussian weight matrix
$$
W_{ij} = \exp\!\left(-\frac{\|r_j-r_i\|^2}{2\ell^2}\right),
$$
where $\ell$ is either chosen directly or calibrated from the original interaction radius $R$. This preserves the idea that nearby agents interact more strongly than distant ones, while making the transition map differentiable.

The function `_effective_gaussian_width_from_radius` chooses $\ell$ so that the Gaussian weight is already small at distance $R$. The function `_get_gaussian_width` then determines which width is used in practice: either a user-specified value or the calibrated value derived from $R$. The function `build_weight_matrix_radius` constructs the corresponding Gaussian interaction matrix under periodic boundary conditions.

The transition function `vk_state_transition` defines the deterministic part of the state evolution. Starting from the current positions and headings, it first computes the Gaussian interaction weights and then forms the alignment drift
$$
\sum_j W_{ij}\sin(\theta_j-\theta_i).
$$
This gives the heading update
$$
\theta_{i,t+1}
=
\theta_{i,t}
+
\beta \, dt \sum_j W_{ij}\sin(\theta_{j,t}-\theta_{i,t}),
$$
followed by angle wrapping. After this, each agent is moved using the updated heading:
$$
r_{i,t+1}
=
r_{i,t}
+
v\,dt
\begin{pmatrix}
\cos(\theta_{i,t+1}) \\
\sin(\theta_{i,t+1})
\end{pmatrix},
$$
and the position is wrapped back into the periodic box.

The function `vk_jacobian_F_approx` computes the Jacobian of this smooth transition map. This is the matrix required by the EKF covariance update. It includes three types of derivatives: derivatives of the heading update with respect to positions through the Gaussian weights, derivatives of the heading update with respect to headings, and derivatives of the position update through the dependence on $\theta_{t+1}$. In particular, because the position update uses the updated heading rather than the old one, the chain rule must be applied when differentiating the motion step.

Overall, this cell replaces the non-differentiable hard-radius dynamics by a smooth approximation that is close in spirit to the original model but suitable for EKF linearisation.

In [ ]:
def _effective_gaussian_width_from_radius(R, boundary_weight=0.05):
    """
    Convert interaction radius R into Gaussian width ell so that
    weight at distance R is small (controlled by boundary_weight).
    """
    if R <= 0:
        raise ValueError("R must be positive.")
    if not (0.0 < boundary_weight < 1.0):
        raise ValueError("boundary_weight must lie in (0,1).")
    return R / np.sqrt(2.0 * np.log(1.0 / boundary_weight))


def _get_gaussian_width(params):
    """
    Return Gaussian kernel width:
    - use params.gaussian_width if provided,
    - otherwise derive from params.R.
    """
    if hasattr(params, "gaussian_width") and params.gaussian_width is not None:
        ell = float(params.gaussian_width)
        if ell <= 0:
            raise ValueError("params.gaussian_width must be positive.")
        return ell

    boundary_weight = getattr(params, "gaussian_boundary_weight", 0.05)
    return _effective_gaussian_width_from_radius(float(params.R), boundary_weight=boundary_weight)


def build_weight_matrix_radius(pos, R, L, include_self=True, weight_mode="binary"):
    """
    Build Gaussian interaction weights under periodic boundaries.

    R is treated as the original interaction scale and mapped to a Gaussian width.
    """
    if weight_mode != "binary":
        raise NotImplementedError("Gaussian EKF uses unnormalised weights only.")

    dx, dy = pairwise_displacements_periodic(pos, L)
    dist2 = dx * dx + dy * dy

    ell = _effective_gaussian_width_from_radius(float(R), boundary_weight=0.05)
    W = np.exp(-dist2 / (2.0 * ell * ell))

    if not include_self:
        np.fill_diagonal(W, 0.0)

    return W


def vk_state_transition(z, params):
    """
    State transition using smooth Gaussian interactions.

    Headings are updated first, then positions are advanced using theta_{t+1}.
    """
    pos, theta = unpack_state(z)

    ell = _get_gaussian_width(params)

    dx, dy = pairwise_displacements_periodic(pos, params.L)
    dist2 = dx * dx + dy * dy
    W = np.exp(-dist2 / (2.0 * ell * ell))

    if not params.include_self:
        np.fill_diagonal(W, 0.0)

    diff = theta[None, :] - theta[:, None]
    drift = (W * np.sin(diff)).sum(axis=1)

    theta_next = wrap_angle(theta + params.beta * params.dt * drift)

    step = params.v * params.dt * np.column_stack([
        np.cos(theta_next),
        np.sin(theta_next)
    ])
    pos_next = wrap_box(pos + step, params.L)

    return pack_state(pos_next, theta_next)


def vk_jacobian_F_approx(z, params):
    """
    Jacobian of the smooth VK transition.

    Includes:
    - derivatives of Gaussian weights wrt positions,
    - full coupling in heading update,
    - chain rule through theta_{t+1} in position update.
    """
    pos, theta = unpack_state(z)
    N = pos.shape[0]

    if params.weight_mode != "binary":
        raise NotImplementedError("Gaussian EKF uses unnormalised weights only.")

    ell = _get_gaussian_width(params)
    dx, dy = pairwise_displacements_periodic(pos, params.L)
    dist2 = dx * dx + dy * dy

    W = np.exp(-dist2 / (2.0 * ell * ell))
    if not params.include_self:
        np.fill_diagonal(W, 0.0)

    diff = theta[None, :] - theta[:, None]
    sin_diff = np.sin(diff)
    cos_diff = np.cos(diff)

    theta_next = np.zeros(N)
    Gx = np.zeros((N, N))
    Gy = np.zeros((N, N))
    Gth = np.zeros((N, N))

    # Compute derivatives of theta update
    for i in range(N):
        drift_i = np.sum(W[i] * sin_diff[i])
        theta_next[i] = wrap_angle(theta[i] + params.beta * params.dt * drift_i)

        for k in range(N):
            dtheta_dxk = 0.0
            dtheta_dyk = 0.0

            for j in range(N):
                delta_ik = 1.0 if i == k else 0.0
                delta_jk = 1.0 if j == k else 0.0

                dW_dxk = W[i, j] * (delta_ik - delta_jk) * dx[i, j] / (ell * ell)
                dW_dyk = W[i, j] * (delta_ik - delta_jk) * dy[i, j] / (ell * ell)

                dtheta_dxk += dW_dxk * sin_diff[i, j]
                dtheta_dyk += dW_dyk * sin_diff[i, j]

            Gx[i, k] = params.beta * params.dt * dtheta_dxk
            Gy[i, k] = params.beta * params.dt * dtheta_dyk

        for k in range(N):
            val = 1.0 if i == k else 0.0
            for j in range(N):
                delta_jk = 1.0 if j == k else 0.0
                delta_ik = 1.0 if i == k else 0.0
                val += params.beta * params.dt * W[i, j] * cos_diff[i, j] * (delta_jk - delta_ik)
            Gth[i, k] = val

    # Assemble full Jacobian
    F = np.zeros((3 * N, 3 * N))

    for i in range(N):
        ix = 3 * i
        iy = 3 * i + 1
        ith = 3 * i + 2

        s = np.sin(theta_next[i])
        c = np.cos(theta_next[i])

        for k in range(N):
            col_xk = 3 * k
            col_yk = 3 * k + 1
            col_thk = 3 * k + 2

            F[ix, col_xk] = (1.0 if i == k else 0.0) + params.v * params.dt * (-s) * Gx[i, k]
            F[ix, col_yk] = params.v * params.dt * (-s) * Gy[i, k]
            F[ix, col_thk] = params.v * params.dt * (-s) * Gth[i, k]

            F[iy, col_xk] = params.v * params.dt * c * Gx[i, k]
            F[iy, col_yk] = (1.0 if i == k else 0.0) + params.v * params.dt * c * Gy[i, k]
            F[iy, col_thk] = params.v * params.dt * c * Gth[i, k]

            F[ith, col_xk] = Gx[i, k]
            F[ith, col_yk] = Gy[i, k]
            F[ith, col_thk] = Gth[i, k]

    return F